# Transform Clickstream
Transforms the clickstream data and saves the cleaned result as a Silver table.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
df = spark.read.table("ecommerce_catalog.bronze.clickstream_raw")
display(df)

In [0]:
df = df.withColumn(
    "is_imputed", 
    F.lit(None).cast("string")
    )
display(df)

In [0]:

from pyspark.sql.types import TimestampType


bedingung_invalid = F.col("event_timestamp") == "INVALID_DATE_TIME"

df = df \
    .withColumn(
        "is_imputed",
        F.when(
            bedingung_invalid,
            F.when(F.col("is_imputed").isNotNull(), F.concat_ws(", ", F.col("is_imputed"), F.lit("event_timestamp")))
             .otherwise(F.lit("event_timestamp"))
        ).otherwise(F.col("is_imputed"))
    ) \
    .withColumn(
        "event_timestamp",
        F.when(bedingung_invalid, F.lit(None))
         .otherwise(F.col("event_timestamp"))
    ) \
    .withColumn(
        "event_timestamp",
        F.regexp_replace(F.col("event_timestamp"), "T", " ")
    ) \
    .withColumn(
        "event_timestamp",
        F.try_to_timestamp(F.col("event_timestamp"), F.lit("yyyy-MM-dd HH:mm:ss"))
    )


display(df)

In [0]:
df = df \
    .withColumn("event_id", F.upper(F.col("event_id"))) \
    .withColumn("event_type", F.lower(F.col("event_type"))) \
    .withColumn("device_type", F.lower(F.trim(F.col("device_type"))))

df = df.withColumn(
    "session_id", 
    F.regexp_replace(F.col("session_id"), "^sess_", "SESS_")
)
display(df)

In [0]:

display(df.filter(
    (F.col("page_url") == "https://shop.example.com/") | 
    (F.col("page_url") == "[https://shop.example.com/](https://www.google.com/search?q=https://shop.example.com/)")
).orderBy(F.col("session_id").asc()))

display(df.filter(
    (F.col("page_url") != "https://shop.example.com/") &
    (F.col("page_url") != "[https://shop.example.com/](https://www.google.com/search?q=https://shop.example.com/)") &
    (F.col("product_id").isNull())
))




         

### **Missing product_id**
The `NULL` values in `product_id` were investigated within the context of the respective sessions and `page_url`. A consistent pattern emerges: missing product IDs occur at the beginning of sessions on generic landing or shop pages, before a specific product has been selected. Only upon subsequently navigating to a product-specific URL is the corresponding `product_id` set and propagated across subsequent product-related events.

The missing values are therefore domain-plausible and do not represent a data quality issue. They are retained unchanged in the Silver layer.

In [0]:
session_ids_list = [
    row["session_id"] 
    for row in df.filter(F.col("customer_id").isNull())
                 .select("session_id")
                 .distinct()
                 .collect()
]

filtered_df = df.filter(F.col("session_id").isin(session_ids_list)) \
                .orderBy(F.col("session_id").desc())

display(filtered_df)

### **Missing customer_id**
Events with missing `customer_id` values were specifically investigated. The affected `event_id` values were identified, and all associated records were reviewed. For these events, no `customer_id` is present across any records, meaning an unambiguous mapping to a known customer is not possible from the available clickstream data.

Reconstructing the missing customer IDs is therefore not reliably feasible. The `NULL` values are intentionally retained in the Silver layer to avoid introducing unsubstantiated customer attributions.

In [0]:

display(df.filter(F.col("browser").isNull()))

display(df.filter(F.col("traffic_source").isNull()))

In [0]:

display(df.filter(F.col("session_id") == "SESS_11i99j"))

display(df.filter(
    (F.col("session_id") == "SESS_20260012") |
    (F.col("session_id") == "SESS_99d44e")
))

In [0]:
session_window = Window.partitionBy("session_id")


df_with_source = df.withColumn(
    "imputed_source",
    F.first("traffic_source", ignorenulls=True).over(session_window)
)


condition_imputiert = F.col("traffic_source").isNull() & F.col("imputed_source").isNotNull()

df = df_with_source \
    .withColumn(
        "is_imputed",
        F.when(
           condition_imputiert,
            F.when(F.col("is_imputed").isNotNull(), F.concat_ws(", ", F.col("is_imputed"), F.lit("traffic_source")))
             .otherwise(F.lit("traffic_source"))
        ).otherwise(F.col("is_imputed"))
    ) \
        .withColumn(
        "traffic_source",
        F.when(condition_imputiert, F.col("imputed_source"))
         .otherwise(F.col("traffic_source"))
    ) \
    .drop("imputed_source")

display(df)

In [0]:

session_window = Window.partitionBy("session_id")


df_with_browser = df.withColumn(
    "imputed_browser",
    F.first("browser", ignorenulls=True).over(session_window)
)


bedingung_browser_imputiert = F.col("browser").isNull() & F.col("imputed_browser").isNotNull()

df = df_with_browser \
    .withColumn(
        "is_imputed",
        F.when(
            bedingung_browser_imputiert,
            F.when(F.col("is_imputed").isNotNull(), F.concat_ws(", ", F.col("is_imputed"), F.lit("browser")))
             .otherwise(F.lit("browser"))
        ).otherwise(F.col("is_imputed"))
    ) \
        .withColumn(
        "browser",
        F.when(bedingung_browser_imputiert, F.col("imputed_browser"))
         .otherwise(F.col("browser"))
    ) \
        .drop("imputed_browser")

display(df)

In [0]:

display(df.filter(F.col("browser").isNull()))

display(df.filter(F.col("traffic_source").isNull()))

In [0]:
df_duplikate = df.groupBy(df.columns) \
                 .count() \
                 .filter(F.col("count") > 1)

display(df_duplikate)

In [0]:
df = df.dropDuplicates()

display(df)

In [0]:
df_duplikate = df.groupBy(df.columns) \
                 .count() \
                 .filter(F.col("count") > 1)

display(df_duplikate)

In [0]:
sparks = SparkSession.builder.getOrCreate()
if sparks.catalog.tableExists("ecommerce_catalog.silver.silver_clickstream"):
   dlt_obj = DeltaTable.forName(spark, "ecommerce_catalog.silver.silver_clickstream")

   dlt_obj.alias("trg").merge(df.alias("src"),"trg.event_id == src.event_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("ecommerce_catalog.silver.silver_clickstream")

In [0]:
%sql
SELECT * FROM ecommerce_catalog.silver.silver_clickstream;

In [0]:
display(df.filter(F.col("is_imputed").isNotNull()))

### **Summary of Silver Transformation**
The clickstream data was cleaned, standardized, and evaluated for domain plausibility for the Silver layer.
The following transformations and data quality decisions were implemented:
- Complete event duplicates were removed.
- `event_timestamp` values were converted from the different source formats into a unified timestamp datatype.
- The invalid timestamp `INVALID_TIME_STAMP` could not be reliably reconstructed and was treated as a missing value.
- Inconsistent casing in `device_type` was unified.
- Inconsistent casing in `event_type` was standardized.
- Inconsistent casing in `event_id` was unified.
- Missing `product_id` values were investigated within the context of the respective sessions and `page_url`. These values occur on generic landing or shop pages before a specific product is selected. They are therefore domain-plausible and were retained unchanged as `NULL`.
- Missing `customer_id` values were specifically investigated. An unambiguous customer attribution could not be reconstructed from the available clickstream data for the affected events. To avoid introducing unsubstantiated mappings, these values were intentionally retained as `NULL`.
- Further missing values that could not be unambiguously reconstructed were not artificially imputed.

The cleaned and standardized clickstream data was subsequently persisted as a Silver table.
